# Fine-tune on Google Colab

This notebook runs the same training pipeline as `task train` but on NVIDIA GPUs (Colab, RunPod, Lambda, etc.) using PyTorch + QLoRA instead of MLX.

**Requirements**: Colab Pro/Pro+ with A100 (40/80 GB) or L4 GPU. Free-tier T4 (16 GB) can only handle 7B models.

| GPU | VRAM | Max Model | Batch | Seq |
|-----|------|-----------|-------|-----|
| T4 | 16 GB | 7B-4bit | 1 | 2048 |
| L4 | 24 GB | 14B-4bit | 1 | 2048 |
| A100 40GB | 40 GB | 32B-4bit | 1 | 4096 |
| A100 80GB | 80 GB | 32B-4bit | 4 | 4096 |

In [ ]:
# 1. Clone the repo (adjust URL to your repo)
!git clone https://github.com/your-username/your-repo.git
%cd your-repo

In [ ]:
# 2. Install CUDA dependencies (replaces the Apple Silicon deps)
!pip install -q \
    torch \
    transformers>=4.45.0 \
    peft>=0.13.0 \
    trl>=0.12.0 \
    bitsandbytes>=0.44.0 \
    datasets>=2.14.0 \
    accelerate>=1.0.0 \
    flash-attn --no-build-isolation \
    pyyaml typer[all] tqdm huggingface_hub[cli]

In [ ]:
# 3. Check GPU
import torch
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} ({props.total_mem / 1e9:.0f} GB VRAM)")
    print(f"Compute capability: {props.major}.{props.minor}")
    print(f"Flash Attention 2: {'yes' if props.major >= 8 else 'no (will use eager attention)'}")
else:
    print("No GPU! Go to Runtime > Change runtime type > GPU")

In [ ]:
# 4. (Optional) Log in to HuggingFace for gated models or pushing results
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 5. Prepare training data (same as `task data` — runs on CPU)
!python src/data/prepare_data.py --config config.yaml

In [ ]:
# 6. Train! Auto-tune picks batch/seq/grad-checkpoint based on your GPU.
!python src/train/train_cuda.py --config config.yaml

In [ ]:
# 7. (Optional) Override auto-tune manually
# !python src/train/train_cuda.py --config config.yaml --batch-size 1 --max-seq-length 2048

In [ ]:
# 8. Push adapters to HuggingFace Hub
# !python src/cli.py push

In [ ]:
# 9. (Optional) Download adapters to use locally with MLX export
# !zip -r /content/adapters.zip output/adapters/
# from google.colab import files
# files.download('/content/adapters.zip')